> ## SOLUTION / ANSWER KEY &mdash; Lab 7.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-assemble-email-agent.ipynb`](../lab-11-assemble-email-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.11 &mdash; Assemble the Email Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Bind gather-only tools into a real Groq agent with create_agent
- Withhold send_email so the agent cannot auto-send
- Run it for real: it gathers & drafts, and you wrap it needs_approval

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; The email-drafting agent, end to end](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-11"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Now assemble the email agent (deck slides 14&ndash;16): `@tool` gather tools bound with
**`create_agent`** (a runnable `CompiledStateGraph`) over a **real Groq model**. The agent gathers
(`lookup_order`, `get_template`) then drafts a reply. The key design choice: the tools list is
**gather-only** &mdash; `send_email` is **defined but not bound** &mdash; and the run's output is
wrapped as a **`needs_approval`** draft. The agent did the tedious 90%; the human keeps the send.

In [ ]:
from langchain_core.tools import tool

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

from langchain_core.tools import tool

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})
@tool
def get_template(kind: str) -> str:
    """Fetch a reply template by kind."""
    return TEMPLATES.get(kind, "")
@tool
def send_email(to: str, body: str) -> str:
    """Send an email. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "SENT"
print("gather tools ready:", lookup_order.name, "&", get_template.name, "| withheld:", send_email.name)

## Build it
The guardrail here is what's **missing**: build the agent with **gather-only** tools (no `send_email`),
and wrap whatever it drafts as a **`needs_approval`** result.

In [ ]:
from langchain.agents import create_agent

def gather_tools():
    return [lookup_order, get_template]

def make_email_agent():
    # bind the REAL Groq model to the gather-only tools
    return create_agent(llm, gather_tools())

def handle_email(draft, tools_used):
    # never auto-send: wrap the drafted reply as a result a human must approve
    return {"draft": draft, "status": "needs_approval", "tools_used": tools_used}

try:
    # Structure checks -- no model call needed:
    print("bound tools :", [t.name for t in gather_tools()])
    print("can it send?:", "send_email" in [t.name for t in gather_tools()])
    print("send_email still EXISTS as a capability, just unbound:", send_email.name)
    demo = handle_email("Hi Priya, your order 4471 is due Friday.", ["lookup_order"])
    print("wrapped     :", demo["status"], "| tools:", demo["tools_used"])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the gather-only agent for real: it looks up the order and drafts a reply &mdash; and it has no send tool, so it cannot auto-send.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        from langchain_core.messages import AIMessage
        def tools_used(messages):
            return [tc["name"] for m in messages for tc in (getattr(m, "tool_calls", None) or [])]

        agent = make_email_agent()
        result = with_backoff(lambda: agent.invoke(
            {"messages": [("user",
                "Look up order 4471, then draft a one-line status reply for the customer. Do not send anything.")]},
            config={"recursion_limit": 8}))
        print_trace(result)
        print("---")
        out = handle_email(result["messages"][-1].content, tools_used(result["messages"]))
        print("tools used:", out["tools_used"])
        print("status    :", out["status"], "(the agent has no send tool, so it cannot auto-send)")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The trace shows **`TOOL CALL: lookup_order`** then a drafted reply &mdash; a real Groq agent doing the gather-then-draft job.
- The bound list is **gather-only**: `send_email` is defined (it exists as a capability) but never passed to `create_agent`, so the agent **cannot send**.
- The output is wrapped **`needs_approval`** &mdash; the agent did the work; a human still holds the send.

## Your turn (open task &mdash; no grader)
Try (carefully) *adding* `send_email` to `gather_tools()` and re-running &mdash; then put it back. **What good
looks like:** you can see that binding the send tool is the ONLY thing standing between "drafts for a human"
and "sends on its own", which is exactly why the guardrail is *withhold the tool*. Restore the gather-only list.

---
### Key takeaway
The guardrail is what's MISSING from the tools list -- send_email is never bound, so the real agent gathers and drafts but cannot send. Next: run the whole pipeline over a suite.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>